In [18]:
# Importing necessary libraries
import pandas as pd  # For data handling
from sklearn.model_selection import train_test_split  # To split data into training/testing sets
from sklearn.pipeline import Pipeline  # To create a pipeline that automates preprocessing + modeling
from sklearn.compose import ColumnTransformer  # To apply different preprocessing to different columns
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # For scaling and encoding
from sklearn.impute import SimpleImputer  # To handle missing values
from sklearn.linear_model import LogisticRegression  # Our classification model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # For evaluation


In [19]:
def load_data(path):
    return pd.read_csv(path)  # Loads the Titanic dataset from a CSV file


In [20]:
def preprocess_and_train(df):
    # Remove unnecessary columns that are not useful for prediction
    df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

    # Separate features (X) and target label (y)
    X = df.drop("Survived", axis=1)
    y = df["Survived"]
        # Numerical features for scaling
    numeric_features = ['Age', 'Fare']

    # Categorical features for encoding
    categorical_features = ['Pclass', 'Sex', 'Embarked', 'SibSp', 'Parch']
        # Pipeline for numerical data: fill missing values with median, then scale
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Pipeline for categorical data: fill missing with mode, then one-hot encode
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))  # handles unseen categories in test data
    ])
        # Combine numeric and categorical pipelines into one preprocessor
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])
        # Complete pipeline: preprocessing followed by logistic regression model
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=200))  # Allow more iterations for convergence
    ])
        # Split the dataset: 80% for training, 20% for testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        # Train the model with the training data
    pipeline.fit(X_train, y_train)
        # Make predictions on the test set
    y_pred = pipeline.predict(X_test)

    # Print accuracy score
    print("Accuracy:", accuracy_score(y_test, y_pred))

    # Detailed report: precision, recall, f1-score
    print("Classification Report:\n", classification_report(y_test, y_pred))

    # Show confusion matrix (true positives/negatives vs false ones)
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



In [22]:
if __name__ == "__main__":
    df = load_data("tested.csv")  # Load dataset
    preprocess_and_train(df)         # Train and evaluate model


Accuracy: 1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        34

    accuracy                           1.00        84
   macro avg       1.00      1.00      1.00        84
weighted avg       1.00      1.00      1.00        84

Confusion Matrix:
 [[50  0]
 [ 0 34]]
